# Notebook for preprocessing
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.R -> dimensionality_reduction.R -> gmm_clustering.ipynb`

In [ ]:
from __future__ import annotations

import itertools
import typing
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from pandas.io.stata import StataReader
from sklearn.model_selection import train_test_split

from utils.equivalent_cat_utils import EquivalenceRule, ImplicationRule, enforce_rules
from utils.preprocessing_utils import (
    convert_data_types,
    create_meta,
    create_missing_info,
    remove_variables,
    sort_groups_meta,
    sort_types_meta,
)
from utils.saving_utils import DatasetBundle

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Read in data
data_path = Path("data/raw/ESA1821_missing.dta")

data_reader = pd.read_stata(data_path, iterator=True, convert_categoricals=False, convert_missing=False)
data_reader = typing.cast(StataReader, data_reader)
value_labels = data_reader.value_labels()
var_labels = data_reader.variable_labels()
with data_reader:
    data_full = data_reader.read()
data_full = DatasetBundle(data_full)

# Set outputs folder
outputs_folder = "data/preprocessed"

# Define discovery and validation data split
# index_discov, _ = train_test_split(
#     data_full.index, test_size=0.5, random_state=np.random.RandomState(np.random.default_rng(3464).bit_generator)
# )
# discov_mask = data_full.index.isin(index_discov)  # Random 50/50 split
discov_mask = data_full.data["zeit"] == 2018  # Discovery data is ESA 2018 and validation data is ESA 2021

In [ ]:
# Create data frame with missingness information
na_labels = {-1: "filter missing", -5: "not included", -9: "information missing"}
data_full = create_missing_info(data_full, na_labels=na_labels)
# data_full.na = cast(pd.DataFrame, data_full.na)

In [ ]:
# Impute using structural rules and drop invalid combinations

# implied by one month abstinence from alcohol
implied_alcohol = {
    "vx210": [0, 1, 2], "wein30gr": 0, "mish30gr": 0, "bier30gr": 0, "bierkons": 0, "weinkons": 0, "spirkons": 0,
    "mishkons": 0, "binge": 1, "alk30gr": 0, "alk30kat": 0,  # TODO: what about audit, hazar, harmf?
}  # fmt: skip

# In the following `implies` is always abstinence from a substance `is_implied` are conclusions that necessarily follow.
rules_alcohol = [
    # one month
    ImplicationRule(implies={"vx210": 0}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alk30gr": 0}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alk30kat": 0}, is_implied=implied_alcohol),
]

rules_nicotine = [
    EquivalenceRule({"hnb30": 0, "typ_hnb": 0}),
    EquivalenceRule({"hnb30": 1, "typ_hnb": 1}),
]

rules_cannabis = [
    # one year
    EquivalenceRule({"hasch_12": 0, "fr12_ha1": 1}),
    # additional
    ImplicationRule(implies={"dsm_ha": 0, "dsm_ha_m": 0}, is_implied={"Miss_Abh_ha": 0}),
    ImplicationRule(implies={"dsm_ha": 1, "dsm_ha_m": 0}, is_implied={"Miss_Abh_ha": 1}),
    ImplicationRule(implies={"dsm_ha": 0, "dsm_ha_m": 1}, is_implied={"Miss_Abh_ha": 1}),
    ImplicationRule(implies={"dsm_ha": 1, "dsm_ha_m": 1}, is_implied={"Miss_Abh_ha": 1}),
]

rules_illegal_drugs = [
    # one year
    EquivalenceRule({"ecsta_12": 0, "fr12_ec1": 1}),
    EquivalenceRule({"lsd_12": 0, "fr12_ls1": 1}),
    EquivalenceRule({"heroi_12": 0, "fr12_he1": 1}),
    EquivalenceRule({"andop_12": 0, "fr12_ao1": 1}),
    EquivalenceRule({"kokai_12": 0, "fr12_ko1": 1}),
    EquivalenceRule({"crack_12": 0, "fr12_cr1": 1}),
    EquivalenceRule({"pilz_12": 0, "fr12_pi1": 1}),
    EquivalenceRule({"nps_12": 0, "fr12_nps1": 1}),
    EquivalenceRule({"amp_12": 0, "fr12_au1": 1}),
    EquivalenceRule({"meth_12": 0, "fr12_meth1": 1}),

    # additional
    ImplicationRule(implies={"dsm_kc": 0, "dsm_kc_m": 0}, is_implied={"Miss_Abh_kc": 0}),
    ImplicationRule(implies={"dsm_kc": 1, "dsm_kc_m": 0}, is_implied={"Miss_Abh_kc": 1}),
    ImplicationRule(implies={"dsm_kc": 0, "dsm_kc_m": 1}, is_implied={"Miss_Abh_kc": 1}),
    ImplicationRule(implies={"dsm_kc": 1, "dsm_kc_m": 1}, is_implied={"Miss_Abh_kc": 1}),

    ImplicationRule(implies={"dsm_au": 0, "dsm_au_m": 0}, is_implied={"Miss_Abh_am": 0}),
    ImplicationRule(implies={"dsm_au": 1, "dsm_au_m": 0}, is_implied={"Miss_Abh_am": 1}),
    ImplicationRule(implies={"dsm_au": 0, "dsm_au_m": 1}, is_implied={"Miss_Abh_am": 1}),
    ImplicationRule(implies={"dsm_au": 1, "dsm_au_m": 1}, is_implied={"Miss_Abh_am": 1}),
]  # fmt: skip

rules_medication = [
    # one year
    ImplicationRule(implies={"anr_12": 0}, is_implied={"m_anr": 0, "t_anr": 0}),
    ImplicationRule(implies={"app_12": 0}, is_implied={"m_app": 0, "t_app": 0}),
    ImplicationRule(implies={"anab_12": 0}, is_implied={"m_anab": 0, "t_anab": 0}),

    # one month
    ImplicationRule(implies={"anr_30": 0}, is_implied={"m_anr": 0, "t_anr": 0}),
    ImplicationRule(implies={"app_30": 0}, is_implied={"m_app": 0, "t_app": 0}),
    ImplicationRule(implies={"anab_30": 0}, is_implied={"m_anab": 0, "t_anab": 0}),
    ImplicationRule(implies={"schm_30": 0}, is_implied={"m_schm": 0, "t_schm": 0}),
    ImplicationRule(implies={"schl_30": 0}, is_implied={"m_schl": 0, "t_schl": 0}),
    ImplicationRule(implies={"ber_30": 0}, is_implied={"m_ber": 0, "t_ber": 0}),
    ImplicationRule(implies={"depr_30": 0}, is_implied={"m_depr": 0, "t_depr": 0}),
    ImplicationRule(implies={"neur_30": 0}, is_implied={"m_neur": 0, "t_neur": 0}),

    # one week
    ImplicationRule(implies={"m_anr": 0}, is_implied={"t_anr": 0}),
    ImplicationRule(implies={"m_app": 0}, is_implied={"t_app": 0}),
    ImplicationRule(implies={"m_anab": 0}, is_implied={"t_anab": 0}),
    ImplicationRule(implies={"m_schm": 0}, is_implied={"t_schm": 0}),
    ImplicationRule(implies={"m_schl": 0}, is_implied={"t_schl": 0}),
    ImplicationRule(implies={"m_ber": 0}, is_implied={"t_ber": 0}),
    ImplicationRule(implies={"m_depr": 0}, is_implied={"t_depr": 0}),
    ImplicationRule(implies={"m_neur": 0}, is_implied={"t_neur": 0}),
]  # fmt: skip

rules = [*rules_alcohol, *rules_nicotine, *rules_cannabis, *rules_illegal_drugs, *rules_medication]
data_full = enforce_rules(data_full, rules=rules, na_strategy="impute", failure_strategy="drop", verbose=True)

In [ ]:
# Create meta-information about variables
data_full = create_meta(data_full, var_labels, value_labels)

# Define variable groupsl
misc_vars = list(data_full.data.columns[:6])
socioecon_vars = list(data_full.data.columns[6:19])
alcohol_vars = list(data_full.data.columns[19:33])
nicotine_vars = list(data_full.data.columns[33:43])
cannabis_vars = list(data_full.data.columns[43:49])
illegal_drugs_vars = list(data_full.data.columns[49:77])
medication_vars = list(data_full.data.columns[77:])

# Assign group names
groups_by_name = {
    "misc": misc_vars, "socioecon": socioecon_vars, "alcohol": alcohol_vars, "nicotine": nicotine_vars,
    "cannabis": cannabis_vars, "illegal_drugs": illegal_drugs_vars, "medication": medication_vars,
}  # fmt: skip

# Define variable types
all_na_vars = [var for var in data_full.data.columns if (data_full.data[var] < 0).all()]

continuous_vars = [
    "wichtn",
    "alter",
    "hh",
    "hh14",
    "wein30gr",
    "mish30gr",
    "bier30gr",
    "audit",
    "alk30gr",
    "Anzahl",
    "pomi",
]

ordinal_vars = [
    "altq", "hne", "vx210", "binge", "alk30kat", "fag_sch", "psecdi_sch", "hnb_fag_sch", "fr12_ha1", "sds_c",
    "fr12_ec1", "fr12_ls1", "fr12_he1", "fr12_ao1", "fr12_ko1", "sds_k", "fr12_cr1", "fr12_pi1", "fr12_nps1",
    "fr12_au1", "sds_a", "fr12_meth1",
]  # fmt: skip

binary_vars = [
    var
    for var in data_full.data.columns
    if var not in itertools.chain(all_na_vars, ordinal_vars, continuous_vars)
    and data_full.data[var].nunique(dropna=True) == 2
    # and set(value_labels.get(var, {}).values()) in [set(["nein", "ja"]), set(["Nein", "Ja"])]
]

nominal_vars = [
    var
    for var in data_full.data.columns
    if var not in itertools.chain(all_na_vars, ordinal_vars, continuous_vars, binary_vars)
]

# Assign type names
types_by_name = {
    "all_na": all_na_vars, "continuous": continuous_vars, "binary": binary_vars, "nominal": nominal_vars,
    "ordinal": ordinal_vars,
}  # fmt: skip

# Add group and type info to information about variables
data_full = sort_groups_meta(data_full, groups_by_name)
data_full = sort_types_meta(data_full, types_by_name)

In [ ]:
# Convert variables to categories (instead of letting StataReader do this, hence convert_categoricals=False in
# pd.read_stata)
data_full = convert_data_types(data_full)
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

In [ ]:
# Recode categories
data_full.data = data_full.data.assign(
    **{
        # Retain the 10 most common cities and lump the others into one category
        # (more than 10 are retained if there are ties)
        "stadt_bl": lambda x: (
            x["stadt_bl"]
            .map(lambda val: val if val in x["stadt_bl"].value_counts().nlargest(n=10, keep="all").index else "other")
            .astype("category")
        ),
    }
)

In [ ]:
# Transform 'labels' column of vars_info into string so it can be serialized as parquet
data_full.meta["labels"] = data_full.meta["labels"].astype("str")

In [ ]:
# Remove variables

# # Variables that are NA everywhere in the discovery or validation dataset
# na_vars: list[str] = (
#     (data_full[discov_mask].isnull().all() | data_full[~discov_mask].isnull().all()).loc[lambda x: x].index.to_list()
# )
# print(
#     f"Dropped the following variables from the dataset because they are missing everywhere in the discovery or "
#     f"validation dataset: {na_vars}"
# )
excluded_vars = [
    # alcohol
    "alk30gr",
    "audit",  # TODO: decide what to do with audit
    "hazar", "harmf",  # TODO: decide how to handle dsm variables

    # nicotine
    "hnb30",
    "dsm_ta",  # TODO: decide how to handle dsm variables
    "fag_sch", "hnb_fag_sch", "psecdi_sch", "Anzahl",  # TODO: hard to deal with, a lot of NAs

    # cannabis
    "hasch_12",
    "sds_c",  # TODO: decide how to handle sds variables
    "dsm_ha", "dsm_ha_m", "Miss_Abh_ha",  # TODO: decide how to handle dsm variables

    # illegal drugs
    "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "sds_k", "crack_12", "pilz_12", "nps_12", "amp_12",
    "sds_a", "meth_12",
    "sds_k", "sds_a",  # TODO: decide how to handle sds variables
    "dsm_kc", "dsm_kc_m", "Miss_Abh_kc", "dsm_au", "dsm_au_m", "Miss_Abh_am",  # TODO: decide how to handle dsm variables

    # medication
    "m_schm", "m_schl", "m_ber", "m_anr", "m_app", "m_depr", "m_neur", "m_anab", "t_schm", "t_schl", "t_ber", "t_anr",
    "t_app", "t_depr", "t_neur", "t_anab",
    "rist", "rist_noa", "pomi",  # TODO: decide what to do with these
    "dsm_schm", "dsm_schm_m", "dsm_schl", "dsm_schl_m", "dsm_ber", "dsm_ber_m",  # TODO: decide how to handle dsm variables
]  # fmt: skip
drop_vars = [*misc_vars, *socioecon_vars, *excluded_vars]  # , *na_vars]
data_reduced = remove_variables(data_full, drop_vars=drop_vars)
data_reduced.meta = cast(pd.DataFrame, data_reduced.meta)
data_reduced.na = cast(pd.DataFrame, data_reduced.na)

In [ ]:
# Discovery/validation split
data_discov_full = DatasetBundle(data_full.data[discov_mask], data_full.meta, data_full.na[discov_mask])
data_valid_full = DatasetBundle(data_full.data[~discov_mask], data_full.meta, data_full.na[~discov_mask])
data_discov_reduced = DatasetBundle(data_reduced.data[discov_mask], data_reduced.meta, data_reduced.na[discov_mask])
data_valid_reduced = DatasetBundle(data_reduced.data[~discov_mask], data_reduced.meta, data_reduced.na[~discov_mask])

In [ ]:
# Save full and reduced data and meta-information about variables
data_full.save(Path(outputs_folder, "data_full"))
data_reduced.save(Path(outputs_folder, "data_reduced"))

# Save full and reduced discovery and validation data
data_discov_full.save(Path(outputs_folder, "data_discov_full"))
data_valid_full.save(Path(outputs_folder, "data_valid_full"))
data_discov_reduced.save(Path(outputs_folder, "data_discov_reduced"))
data_valid_reduced.save(Path(outputs_folder, "data_valid_reduced"))